# RML Preprocessing Script
This script is used for generating RML_RAW_PREPROCESSED data from the raw RAW data, you can download the raw dataset from: https://www.kaggle.com/datasets/ryersonmultimedialab/ryerson-emotion-database

emotion: {'ang': 0, 'exc': 1, 'fru': 2, 'hap': 3, 'neu': 4, 'sad': 5}

In [1]:
import os, sys
import glob
import pickle
import numpy as np
import pandas as pd
import cv2
from scipy.io import wavfile
from tqdm import tqdm
import torch
from facenet_pytorch import MTCNN

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=48, margin=2, post_process=False, device=device)

# Common Functions

In [3]:
def read_video(file_name):
    vidcap = cv2.VideoCapture(file_name)
    
    # Read FPS
    (major_ver, minor_ver, subminor_ver) = (cv2.__version__).split('.')
    if int(major_ver)  < 3 :
        fps = vidcap.get(cv2.cv.CV_CAP_PROP_FPS)
    else :
        fps = vidcap.get(cv2.CAP_PROP_FPS)
    
    # Read image data
    success, image = vidcap.read()
    images = []
    while success:
        images.append(image)
        success, image = vidcap.read()
    return np.stack(images), fps

def dump_image(img_segment, out_path='./'):
    count = 0
    for i in range(img_segment.shape[0]):
        faces = mtcnn(img_segment[i,:,:,:])
        if faces != None:
#             cv2.imwrite(f'{out_path}/image_{i}.jpg', img_segment[i,:,:,:])
            cv2.imwrite(f'{out_path}/image_{count}.jpg', faces.permute(1, 2, 0).int().numpy())
            count = count + 1
#             mtcnn(img_segment[i,:,:,:], save_path=f'{out_path}/image_{i}.jpg')

In [4]:
%%time
# Process multimodal data over all sessions
# NOTE: This might take several hours to run, the time listed on this cell is for processing 5 label files
output_path = r'/home/matt/Model/Database_processed/RML_RAW_PROCESSED_Face'
#     list wav file
if not os.path.exists(output_path):
    os.makedirs(output_path)
wav_path = r'/home/matt/Model/Data/RML/audio_file'
wav_list = os.listdir(wav_path)
for i in range(len(wav_list)):
    wav_list[i] = wav_list[i][:-4]

    
all_metas = {}
for base_path in glob.glob(r'/home/matt/Model/Data/RML/archive/*'):
    for root, dirs, files in os.walk(base_path, topdown=False):
        for name in files:
            print(os.path.join(root, name))
            if name[-3:] == 'avi':
#                 print('1', os.path.join(root, name))
                audio_file_name = '_'.join(os.path.join(root, name).split('/')[8:])
                # print('2', audio_file_name)
                index = wav_list.index(audio_file_name[:-4])
                sr, signal  = wavfile.read(os.path.join(wav_path, wav_list[index]) + '.wav')
                images, fps = read_video(os.path.join(root, name))
#                 print('3', os.path.join(wav_path, wav_list[index]) + '.wav')
                out_path = os.path.join(output_path, audio_file_name[:-4])
    #             print(out_path)
                if not os.path.exists(out_path):
                    os.makedirs(out_path)
                wavfile.write(f'{out_path}/audio.wav', sr, signal)
                dump_image(images, out_path)

/home/matt/Model/Data/RML/archive/s1/s1/su1.avi
/home/matt/Model/Data/RML/archive/s1/s1/ha1.avi
/home/matt/Model/Data/RML/archive/s1/s1/fe5.avi
/home/matt/Model/Data/RML/archive/s1/s1/di3.avi
/home/matt/Model/Data/RML/archive/s1/s1/an2.avi
/home/matt/Model/Data/RML/archive/s1/s1/su4.avi
/home/matt/Model/Data/RML/archive/s1/s1/sa4.avi
/home/matt/Model/Data/RML/archive/s1/s1/di1.avi
/home/matt/Model/Data/RML/archive/s1/s1/fe2.avi
/home/matt/Model/Data/RML/archive/s1/s1/fe4.avi
/home/matt/Model/Data/RML/archive/s1/s1/sa5.avi
/home/matt/Model/Data/RML/archive/s1/s1/ha3.avi
/home/matt/Model/Data/RML/archive/s1/s1/Thumbs.db
/home/matt/Model/Data/RML/archive/s1/s1/su5.avi
/home/matt/Model/Data/RML/archive/s1/s1/su2.avi
/home/matt/Model/Data/RML/archive/s1/s1/an1.avi
/home/matt/Model/Data/RML/archive/s1/s1/fe3.avi
/home/matt/Model/Data/RML/archive/s1/s1/sa3.avi
/home/matt/Model/Data/RML/archive/s1/s1/di2.avi
/home/matt/Model/Data/RML/archive/s1/s1/ha2.avi
/home/matt/Model/Data/RML/archive/s1/s

In [5]:
df = pd.read_csv('/home/matt/Model/Data/RML/text.csv')
df

,filename,text
0,s1_an1.avi,Vattene sei talmente stupido!
1,s1_an2.avi,"Stami lontano, coglione!"
2,s1_an3.avi,Ma di cosa cazzo stai parlando?
3,s1_di1.avi,... ... ... ... ...
4,s1_di2.avi,Quatro peças. Anava sei lá quê vo mushrooms e...
...,...,...
715,s8_f3chi_sa4.avi,"還沒趕上車,我遲到了"
716,s8_f3chi_sa5.avi,NaN
717,s8_f3chi_su1.avi,什麼?我中獎了?
718,s8_f3chi_su2.avi,"天哪,到底怎么回事啊?"


In [6]:
metadata = {}
emo_trans = {'ha': 'hap', 'sa': 'sad', 'an': 'ang', 'fe': 'fea', 'su': 'sur', 'di': 'dis'}
df = pd.read_csv(r'/home/matt/Model/Data/RML/text.csv')
for index, row in df.iterrows():
    print(row['filename'])
    print(row['text'])
    emo = row['filename'].split('_')[-1][:2]
    metadata[row['filename'][:-4]] = {'text': row['text'], 'label': emo_trans[emo]}

pickle.dump(metadata, open(r'/home/matt/Model/Database_processed/RML_RAW_PROCESSED_Face/meta.pkl','wb'))

s1_an1.avi
 Vattene sei talmente stupido!
s1_an2.avi
 Stami lontano, coglione!
s1_an3.avi
 Ma di cosa cazzo stai parlando?
s1_di1.avi
 ... ... ... ... ...
s1_di2.avi
 Quatro peças. Anava sei lá quê vo mushrooms eu Other
s1_di3.avi
 S48 Man, to bih je pulca.
s1_di4.avi
 ... ... ... ... ... ...
s1_fe1.avi
 No, non mi uccidere!
s1_fe2.avi
 No, per favore, non mi ammazzolare!
s1_fe3.avi
 è così buio qui
s1_fe4.avi
RISAS
s1_fe5.avi
 Oh mio Dio, ho una paura!
s1_ha1.avi
 sono troppo contento oggi
s1_ha2.avi
 mi hanno preso all'università!
s1_ha3.avi
 Mi piace un sacco questo paese!
s1_sa1.avi
 Ma porca Eva, mi hanno bruciato alle zan.
s1_sa2.avi
 Ho perso l'ultimo polmo.
s1_sa3.avi
 ho perso tutti i soldi
s1_sa4.avi
 Ho fatto una cazzata all'esonero.
s1_sa5.avi
 il mio cane è morto
s1_su1.avi
 Cosa? O vindo eu?
s1_su2.avi
 veramente? ma non ci credo!
s1_su3.avi
 Cosa? Mas é verdade?
s1_su4.avi
 veramente? ma sei sicuro?
s1_su5.avi
 Ozzio fama cosa è successo?
s2_an1.avi
 Vieni da qui, stupid

In [7]:
import pickle
with open(r'/home/matt/Model/Database_processed/RML_RAW_PROCESSED_Face/meta.pkl', 'rb') as f:
    data = pickle.load(f)

In [8]:
data

{'s1_an1': {'text': ' Vattene sei talmente stupido!', 'label': 'ang'},
 's1_an2': {'text': ' Stami lontano, coglione!', 'label': 'ang'},
 's1_an3': {'text': ' Ma di cosa cazzo stai parlando?', 'label': 'ang'},
 's1_di1': {'text': ' ... ... ... ... ...', 'label': 'dis'},
 's1_di2': {'text': ' Quatro peças. Anava sei lá quê vo mushrooms eu Other',
  'label': 'dis'},
 's1_di3': {'text': ' S48 Man, to bih je pulca.', 'label': 'dis'},
 's1_di4': {'text': ' ... ... ... ... ... ...', 'label': 'dis'},
 's1_fe1': {'text': ' No, non mi uccidere!', 'label': 'fea'},
 's1_fe2': {'text': ' No, per favore, non mi ammazzolare!', 'label': 'fea'},
 's1_fe3': {'text': ' è così buio qui', 'label': 'fea'},
 's1_fe4': {'text': 'RISAS', 'label': 'fea'},
 's1_fe5': {'text': ' Oh mio Dio, ho una paura!', 'label': 'fea'},
 's1_ha1': {'text': ' sono troppo contento oggi', 'label': 'hap'},
 's1_ha2': {'text': " mi hanno preso all'università!", 'label': 'hap'},
 's1_ha3': {'text': ' Mi piace un sacco questo paese!